# Multi-Qubit Gates (CNOT, CZ, SWAP, Toffoli)

## Single-qubit gates can only rotate each qubit independently — they cannot create entanglement. For that we need gates that act on two or more qubits simultaneously. These are the gates that put the *quantum* in quantum computing.

# 1. CNOT (controlled-NOT)

## The workhorse two-qubit gate. With one **control** qubit and one **target** qubit:

## - If the control is $|0\rangle$, do nothing to the target.
## - If the control is $|1\rangle$, apply $X$ to the target.

## In matrix form on the basis $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$:

$$ \Large  \text{CNOT} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}. $$

In [ ]:
import numpy as np

CNOT = np.array([[1,0,0,0],
                 [0,1,0,0],
                 [0,0,0,1],
                 [0,0,1,0]], dtype=complex)

# Verify the truth table
labels = ['|00>','|01>','|10>','|11>']
for i, lab in enumerate(labels):
    v = np.zeros(4, dtype=complex); v[i] = 1
    j = int(np.argmax(np.abs(CNOT @ v)))
    print(f'CNOT {lab} = {labels[j]}')

# 2. Creating entanglement: H then CNOT

## The two-step recipe for the Bell state $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$:

$$ \Large  |00\rangle \;\xrightarrow{H \otimes I}\; \tfrac{1}{\sqrt{2}}(|00\rangle + |10\rangle) \;\xrightarrow{\text{CNOT}}\; \tfrac{1}{\sqrt{2}}(|00\rangle + |11\rangle). $$

## Apply Hadamard to the control qubit to create superposition, then CNOT to entangle the two.

In [ ]:
H = (1/np.sqrt(2)) * np.array([[1,1],[1,-1]], dtype=complex)
I = np.eye(2, dtype=complex)

ket00 = np.array([1,0,0,0], dtype=complex)

step1 = np.kron(H, I) @ ket00
bell  = CNOT @ step1
print('After H on q0:   ', step1)
print('After CNOT:      ', bell)
print('= |Phi+>?        ', np.allclose(bell, np.array([1,0,0,1])/np.sqrt(2)))

# 3. Controlled-Z (CZ)

$$ \Large  \text{CZ} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & -1 \end{pmatrix} = \text{diag}(1, 1, 1, -1). $$

## CZ is **symmetric** in its two qubits — it doesn't distinguish control from target — and only flips the phase of $|11\rangle$. It is the natural two-qubit gate on many platforms (Rydberg atoms, some superconductors). CZ and CNOT are related by Hadamards: $\text{CNOT} = (I \otimes H)\, \text{CZ}\, (I \otimes H)$.

In [ ]:
CZ = np.diag([1, 1, 1, -1]).astype(complex)

lhs = np.kron(I, H) @ CZ @ np.kron(I, H)
print('(I o H) CZ (I o H) = CNOT ?', np.allclose(lhs, CNOT))

# 4. SWAP

$$ \Large  \text{SWAP} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \end{pmatrix}. $$

## Exchanges the states of two qubits: $\text{SWAP}|a, b\rangle = |b, a\rangle$. It is **not** primitive on most hardware — it is usually compiled into three CNOTs:

$$ \Large  \text{SWAP} = \text{CNOT}_{12}\,\text{CNOT}_{21}\,\text{CNOT}_{12}. $$

In [ ]:
SWAP = np.array([[1,0,0,0],
                 [0,0,1,0],
                 [0,1,0,0],
                 [0,0,0,1]], dtype=complex)

# Build SWAP from three CNOTs
CNOT_12 = CNOT
CNOT_21 = np.array([[1,0,0,0],
                    [0,0,0,1],
                    [0,0,1,0],
                    [0,1,0,0]], dtype=complex)  # control on q1, target q0

built = CNOT_12 @ CNOT_21 @ CNOT_12
print('SWAP from 3 CNOTs ?', np.allclose(built, SWAP))

# 5. Toffoli (CCNOT)

## A three-qubit gate: flip the target iff both controls are $|1\rangle$. Equivalent to the classical AND gate, but reversible. Toffoli alone is **universal for classical reversible computation**, so any classical circuit can be embedded in a quantum circuit.

$$ \Large  \text{Toffoli}|a,b,c\rangle = |a, b, c \oplus ab\rangle. $$

## On real hardware Toffoli decomposes into 6 CNOTs plus single-qubit gates — it is not a hardware-native operation.

In [ ]:
# Build Toffoli as an 8x8 matrix
Toffoli = np.eye(8, dtype=complex)
Toffoli[[6, 7]] = Toffoli[[7, 6]]   # swap the last two basis states
print('Toffoli unitary?', np.allclose(Toffoli.conj().T @ Toffoli, np.eye(8)))
print('Toffoli |110> -> |111> ?')
v = np.zeros(8); v[6] = 1
print(Toffoli @ v)

# Recap

## - **CNOT**: flip target iff control is $|1\rangle$. The canonical entangling gate.
## - **CZ**: phase-flip $|11\rangle$ only; symmetric in its qubits.
## - **SWAP** = three CNOTs.
## - **Toffoli (CCNOT)**: three-qubit gate, universal for classical reversible computing.

## Next: what does it mean for a gate set to be *universal*?